# 01 — Augmentation Lab

**Purpose:** Visualize and validate what the training dataloader actually
emits. Covers both the YOLOP-baseline augmentations (letterbox +
`random_perspective` + HSV + flip) and the YOLOPv2-style additions
(Mosaic + MixUp, [INFERRED from YOLOv7 recipe]). The final cell also
dumps a preview grid back to Drive for downstream auditing.

**Checks:**
- Letterbox resizing and padding
- Random perspective + HSV + flip alignment (img ↔ lane ↔ boxes)
- Mosaic 4-tile composition
- MixUp blend (image + label concat)
- Bbox label normalization consistency

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
from lib.config import cfg
from lib.dataset import BddDataset
from lib.utils.drive_dataset import ensure_local_dataset_from_drive, find_raw_bdd_root

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)

cfg.defrost()
cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = os.path.join(RAW_BDD_ROOT, 'images', '100k')
cfg.DATASET.LABELROOT = os.path.join(RAW_BDD_ROOT, 'labels', '100k')
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')
cfg.freeze()

train_dataset = BddDataset(cfg, is_train=True, inputsize=640, transform=T.ToTensor())
val_dataset = BddDataset(cfg, is_train=False, inputsize=640, transform=T.ToTensor())
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')


In [ ]:
# ── Visualize augmented samples ──
from lib.utils import xyxy2xywh

def show_sample(dataset, idx, ax_img, ax_lane, title=''):
    img, target, path, shapes = dataset[idx]
    det_labels, lane_label = target
    
    # Image: CHW -> HWC
    img_np = img.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    
    # Draw det boxes (labels are in xywh normalized)
    h, w = img_np.shape[:2]
    img_draw = (img_np * 255).astype(np.uint8).copy()
    for label in det_labels:
        _, cls, cx, cy, bw, bh = label.numpy()
        if bw == 0:
            continue
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_draw, f'{int(cls)}', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    
    ax_img.imshow(img_draw)
    ax_img.set_title(f'{title} img ({det_labels.shape[0]} dets)')
    ax_img.axis('off')
    
    # Lane mask: 2-ch -> argmax
    lane_np = lane_label.numpy()
    lane_vis = lane_np[1]  # foreground channel
    ax_lane.imshow(lane_vis, cmap='gray')
    ax_lane.set_title(f'{title} lane')
    ax_lane.axis('off')

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
indices = np.random.choice(len(train_dataset), 4, replace=False)
for row, idx in enumerate(indices):
    # Train (augmented)
    show_sample(train_dataset, idx, axes[row, 0], axes[row, 1], f'Train[{idx}]')
    # Val (no augmentation)
    val_idx = min(idx, len(val_dataset)-1)
    show_sample(val_dataset, val_idx, axes[row, 2], axes[row, 3], f'Val[{val_idx}]')
plt.suptitle('Augmented (train) vs Clean (val) — Boxes + Lane Masks', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Augmentation consistency check ──
# Same sample loaded multiple times should produce different augmentations
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
idx = 0
for col in range(5):
    show_sample(train_dataset, idx, axes[0, col], axes[1, col], f'Aug #{col}')
plt.suptitle(f'Same sample (idx={idx}) — 5 different augmentations', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Batch collate check ──
from torch.utils.data import DataLoader

loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                    num_workers=0, collate_fn=train_dataset.collate_fn)
img, target, paths, shapes = next(iter(loader))
det_labels, lane_labels = target

print(f'Batch img:   {img.shape}')          # [8, 3, 640, 640]
print(f'Det labels:  {det_labels.shape}')    # [N, 6]  (img_idx, cls, x, y, w, h)
print(f'Lane labels: {lane_labels.shape}')   # [8, 2, 640, 640]
print(f'Det label range: cls={det_labels[:,1].unique().tolist()}')
print(f'Lane label range: min={lane_labels.min():.3f}, max={lane_labels.max():.3f}')
print('Collate check passed!')

In [ ]:
# ── Mosaic + MixUp visualization (the real dataloader path) ──
# Rebuild the dataset with MOSAIC + MIXUP = True so what we plot is
# literally what training sees when YOLOPv2 baseline is active.
cfg.defrost()
cfg.DATASET.MOSAIC = True
cfg.DATASET.MOSAIC_PROB = 1.0
cfg.DATASET.MIXUP = True
cfg.DATASET.MIXUP_PROB = 1.0  # forced ON for visualization
cfg.freeze()

mosaic_dataset = BddDataset(cfg, is_train=True, inputsize=640, transform=T.ToTensor())

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for row in range(3):
    for col in range(4):
        idx = np.random.randint(len(mosaic_dataset))
        show_sample(mosaic_dataset, idx, axes[row, col], axes[row, col], f'M[{idx}]')
        # second pass to separately show lane — use a dedicated cell below if needed
plt.suptitle('Mosaic + MixUp samples (YOLOPv2 training path, [INFERRED])', fontsize=14)
plt.tight_layout()
plt.show()

# Save a preview grid back to Drive for audit/review.
import pathlib
out_dir = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/metrics/augmentation_preview'
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(os.path.join(out_dir, 'mosaic_mixup_grid.png'), dpi=120, bbox_inches='tight')
print(f'Saved preview grid -> {out_dir}/mosaic_mixup_grid.png')